# Notebook 02 — Data Cleaning & Exploratory Analysis

**Phase 2 learning checkpoint.**  In Notebook 01 we *met* the raw data. Here we *clean* it and then *interrogate* it.

## What you will do here

1. Re-load the raw Ames data.
2. Walk through each cleaning step in `src/preprocessing.py`, peeking before and after so you can see what changes.
3. Run a battery of EDA helpers from `src/eda.py` on the cleaned data.
4. Save the clean DataFrame to `data/processed/ames_clean.csv` so Phase 3 / 4 don't have to re-clean it.
5. Write down your observations at the end — what features look promising? What surprised you?

## Reading order

Pair this notebook with [`docs/03_data_cleaning_explained.md`](../docs/03_data_cleaning_explained.md) (cleaning theory) and [`docs/04_exploratory_data_analysis.md`](../docs/04_exploratory_data_analysis.md) (EDA theory).

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

from src.data_loader import load_ames
from src import preprocessing as pp
from src import eda

print("Setup complete.")

## 1. The raw data, one more time

Before cleaning, eyeball the issues we are about to fix.

In [ ]:
raw = load_ames()
print(f"shape: {raw.shape}")
print(f"NaN cells: {raw.isna().sum().sum():,}")
print(f"object/categorical columns: {raw.select_dtypes(include='object').shape[1]}")
print(f"sample column names (with spaces): {raw.columns[:6].tolist()}")
raw.head(3)

## 2. Cleaning step-by-step

We'll walk through each function in `preprocessing.py` so you can see exactly what each one does — instead of running `clean_ames()` and trusting it.

### 2.1 Standardize column names

Goal: `"MS SubClass"` → `"ms_subclass"`. This eliminates spaces and uppercase, so the rest of the code can use clean Python-friendly names.

In [ ]:
df = pp.standardize_column_names(raw)
print("Before:", raw.columns[:6].tolist())
print("After: ", df.columns[:6].tolist())

### 2.2 Drop identifier columns

`pid` (parcel ID) and `order` (row index) uniquely identify each home. They have no predictive value — and worse, leaving them in lets a model 'cheat' by memorizing the lookup. We drop them now.

In [ ]:
before_cols = df.shape[1]
df = pp.drop_identifier_columns(df)
print(f"Dropped {before_cols - df.shape[1]} identifier column(s); now {df.shape[1]} columns.")

### 2.3 Remove known outliers (De Cock, 2011)

Five homes with `gr_liv_area > 4000` are partial sales or weirdly cheap mansions. The dataset's author specifically recommends removing them when fitting a regression model. Notice we do **not** generically remove statistical outliers — we only remove what *domain knowledge* tells us is unrepresentative.

In [ ]:
before_rows = df.shape[0]
df = pp.remove_known_outliers(df)
print(f"Rows: {before_rows} -> {df.shape[0]}")

### 2.4 Fill structural NAs

In Ames, many `NaN` values are not 'unknown' — they're 'doesn't have one'. `pool_qc = NaN` means the home has no pool. We replace those with the literal `"None"` (becomes its own one-hot category later) so the model doesn't lose that signal.

In [ ]:
before_nan = df.isna().sum().sum()
df = pp.fill_structural_nas(df)
after_nan = df.isna().sum().sum()
print(f"NaN cells: {before_nan:,} -> {after_nan:,}")
print("\npool_qc value counts (the 'None' here are our sentinel for 'no pool'):")
print(df["pool_qc"].value_counts(dropna=False).head())

### 2.5 Impute remaining (truly missing) values

Whatever NaN survived step 2.4 is *actual* unknown data — for example, a couple of `lot_frontage` values are genuinely missing. We fill numeric columns with the median (robust to outliers) and categorical columns with the mode.

In [ ]:
df = pp.impute_missing(df)
print(f"NaN cells after imputation: {df.isna().sum().sum()}")

### 2.6 Encode ordinals

Columns like `kitchen_qual` are *ordinal*: their values are ordered (Ex > Gd > TA > Fa > Po). We map them to integers preserving that order. After this step, the column is numeric and ready for any model.

In [ ]:
before = df["kitchen_qual"].head(8).tolist()
df = pp.encode_ordinals(df)
after = df["kitchen_qual"].head(8).tolist()
print("kitchen_qual sample:")
print("  before:", before)
print("  after: ", after)

### 2.7 One-hot encode nominals

Columns like `neighborhood` are *nominal* — no inherent order. We turn one such column into many 0/1 indicator columns (one per category). `drop_first=True` drops one to avoid multicollinearity.

In [ ]:
before_cols = df.shape[1]
df = pp.encode_nominals(df)
print(f"Columns: {before_cols} -> {df.shape[1]} (added {df.shape[1] - before_cols} one-hot dummies)")
print("\nFirst 5 neighborhood dummy columns:")
print([c for c in df.columns if c.startswith("neighborhood_")][:5])

### Sanity check: are we model-ready?

After all that, we want: zero NaN, all-numeric columns, and a tidy `(rows, features)` matrix.

In [ ]:
print(f"shape: {df.shape}")
print(f"NaN cells: {df.isna().sum().sum()}")
print(f"non-numeric columns left: {df.select_dtypes(exclude=np.number).shape[1]}")
print("\ndtype counts:")
print(df.dtypes.value_counts())

## 3. EDA on the clean data

Now that the data is in shape, we can interrogate it. The goal is to *build intuition*: which features look like they will matter, and what does the target distribution look like?

### 3.1 Numeric summary, sorted by absolute skewness

Skewness > 1 means a noticeably lopsided distribution. Strongly skewed *features* often benefit from log-transformation (Phase 3). Strongly skewed *targets* almost always do.

In [ ]:
eda.numeric_summary(df).head(12).round(2)

### 3.2 Target distribution

`saleprice` is right-skewed (long tail of expensive homes). `log1p(saleprice)` is much closer to a bell curve — that often makes regression easier.

In [ ]:
eda.plot_distribution(df["saleprice"], log=True)

### 3.3 Top features by absolute correlation with the target

A first, fast 'which features matter?' question. **Pearson** correlation measures *linear* relationships only — so this misses nonlinear patterns. Still, it's a great starting filter.

In [ ]:
top = eda.top_correlations(df, target="saleprice", n=20)
top.round(3)

### 3.4 Correlation heatmap of the top features

When two features are very correlated with each other (not just with the target), they carry redundant information. Linear models hate that; trees are fine with it. Watch for big red or big blue squares far from the diagonal.

In [ ]:
eda.plot_correlation_heatmap(df, target="saleprice", n=12)

### 3.5 The two strongest predictors, visually

Look at the actual scatter plots — correlation hides shape. Are the points tightly hugging the line, or noisy? Any heteroskedasticity (more spread at the high end)?

In [ ]:
for feat in top.index[:2]:
    eda.plot_numeric_vs_target(df, feat, target="saleprice", log_y=True)

### 3.6 A categorical feature view — Overall Quality

`overall_qual` is the dataset's own 1–10 rating. We expect a clean ranking from low to high.

We use the **raw** Ames data here (before one-hot) so the column exists as an ordinal, not as a wall of dummies.

In [ ]:
# Re-run cleaning with one_hot=False so categorical/ordinal columns stay intact for plotting.
df_for_box = pp.clean_ames(load_ames(), one_hot=False)
eda.plot_categorical_vs_target(df_for_box, feature="overall_qual", target="saleprice")

## 4. Save the cleaned data

Phase 3 and beyond will start from this file, not from the raw download. Saving it to `data/processed/` is a small but critical reproducibility habit.

In [ ]:
out_path = PROJECT_ROOT / "data" / "processed" / "ames_clean.csv"
df.to_csv(out_path, index=False)
print(f"Wrote {df.shape[0]:,} rows x {df.shape[1]:,} cols -> {out_path}")

## 5. Wrap-up — what did you learn?

Write your answers down before moving on:

1. **Counting the surgery.** How many NaN cells did we start with? How many after step 2.4? How many at the end? Where did most of them disappear?
2. **Why the order matters.** Why did we fill structural NAs *before* impute_missing? What would have gone wrong otherwise?
3. **The top 5 predictors.** Which 5 features came out on top in section 3.3? Do they 'feel right' to you for a real estate price model? Any surprises?
4. **Linear vs. nonlinear.** Look at the scatter plots in 3.5 — is the relationship clean and straight, or does it bend / fan out? What might that mean for our choice of model in Phase 4?
5. **Ordinal vs. one-hot.** Reading the columns of the final DataFrame: which ones are now integers because they were ordinal? Which ones became many `*_*` dummy columns because they were nominal?

When you can answer these clearly, you're ready for **Phase 3: Feature Engineering**.